Derm7pt -- Backbone Sweep -- LIGHTWEIGHT TIER ONLY (run on one of your 3 Kaggle accounts in parallel)

Before running:

Upload derm7pt_release_v0.zip (or the extracted release_v0 folder) as a private Kaggle Dataset on your main account, then add your other 2 accounts as collaborators on that dataset (Dataset page -> Settings -> Sharing -> Add collaborator).
On this account, attach the shared dataset to this notebook.
Notebook settings -> Accelerator: GPU T4 x2 (or whatever dual-GPU option you have). Internet: On (needed to download ImageNet-pretrained weights via timm).
Edit INPUT_DIR in the config cell below to match the attached dataset's path (check the Kaggle "Input" sidebar for the exact path).
This notebook only trains the lightweight tier. Run the other two tiers on your other 2 Kaggle accounts at the same time. After this finishes, merge sweep_results_lightweight.csv with the medium and heavy tier results (from your other 2 accounts) to build the full leaderboard.

In [ ]:
import os, time, csv, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score

try:
    import timm
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' -', torch.cuda.get_device_name(i))

torch 2.11.0+cu128 | cuda available: True | GPUs: 2
 - Tesla T4
 - Tesla T4


In [ ]:
# ---- CONFIG: edit INPUT_DIR to match your attached Kaggle dataset ----
INPUT_DIR = '/kaggle/input/datasets/noorabdullah02/derm7pt-release-v0/release_v0'  # <-- EDIT ME (check the Input sidebar)
OUTPUT_DIR = '/kaggle/working'
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_PER_BACKBONE = 15
LR = 3e-4
NUM_WORKERS = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MULTI_GPU = torch.cuda.device_count() > 1
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)

# If you attached the raw .zip instead of a pre-extracted folder, unzip it once:
if not os.path.isdir(INPUT_DIR):
    zip_candidates = [p for p in os.listdir('/kaggle/input') if p.endswith('.zip')]
    print('INPUT_DIR not found as a folder. Zip files seen in /kaggle/input:', zip_candidates)
    print('Either point INPUT_DIR at the extracted folder, or unzip manually into', OUTPUT_DIR)

In [ ]:
# ---- labels.py ----
DIAGNOSIS_GROUPS = {
    'basal cell carcinoma': 'basal cell carcinoma', 'blue nevus': 'blue nevus',
    'clark nevus': 'clark nevus', 'combined nevus': 'combined nevus',
    'congenital nevus': 'congenital nevus', 'dermal nevus': 'dermal nevus',
    'dermatofibroma': 'dermatofibroma', 'lentigo': 'lentigo',
    'melanoma': 'melanoma', 'melanoma (in situ)': 'melanoma',
    'melanoma (less than 0.76 mm)': 'melanoma', 'melanoma (0.76 to 1.5 mm)': 'melanoma',
    'melanoma (more than 1.5 mm)': 'melanoma', 'melanoma metastasis': 'melanoma',
    'melanosis': 'melanosis', 'miscellaneous': 'miscellaneous',
    'recurrent nevus': 'recurrent nevus', 'reed or spitz nevus': 'reed or spitz nevus',
    'seborrheic keratosis': 'seborrheic keratosis', 'vascular lesion': 'vascular lesion',
}
DIAGNOSIS_CLASSES = sorted(set(DIAGNOSIS_GROUPS.values()))
MALIGNANT_DIAGNOSES = {'melanoma', 'basal cell carcinoma'}

SEVEN_POINT_CRITERIA = {
    'pigment_network': ['absent', 'typical', 'atypical'],
    'blue_whitish_veil': ['absent', 'present'],
    'vascular_structures': ['absent', 'arborizing', 'comma', 'hairpin', 'within regression', 'wreath', 'dotted', 'linear irregular'],
    'pigmentation': ['absent', 'diffuse regular', 'localized regular', 'diffuse irregular', 'localized irregular'],
    'streaks': ['absent', 'regular', 'irregular'],
    'dots_and_globules': ['absent', 'regular', 'irregular'],
    'regression_structures': ['absent', 'blue areas', 'white areas', 'combinations'],
}
CRITERIA_NUM_CLASSES = {k: len(v) for k, v in SEVEN_POINT_CRITERIA.items()}

def diagnosis_to_label(d):
    return DIAGNOSIS_GROUPS[d.strip().lower()]

def diagnosis_to_malignant(d):
    return int(diagnosis_to_label(d) in MALIGNANT_DIAGNOSES)

def criterion_to_index(col, value):
    return SEVEN_POINT_CRITERIA[col].index(value.strip().lower())

In [ ]:
# ---- dataset.py ----
CRITERIA_COLUMNS = list(SEVEN_POINT_CRITERIA.keys())

class Derm7ptDataset(Dataset):
    def __init__(self, release_dir, split, transform=None):
        assert split in ('train', 'valid', 'test')
        meta_dir = os.path.join(release_dir, 'meta')
        self.images_dir = os.path.join(release_dir, 'images')
        self.transform = transform
        meta = pd.read_csv(os.path.join(meta_dir, 'meta.csv'))
        split_idx = pd.read_csv(os.path.join(meta_dir, f'{split}_indexes.csv'))['indexes']
        self.df = meta.iloc[split_idx].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clinic_img = Image.open(os.path.join(self.images_dir, row['clinic'])).convert('RGB')
        derm_img = Image.open(os.path.join(self.images_dir, row['derm'])).convert('RGB')
        if self.transform is not None:
            clinic_img = self.transform(clinic_img)
            derm_img = self.transform(derm_img)
        diag_label = DIAGNOSIS_CLASSES.index(diagnosis_to_label(row['diagnosis']))
        malignant_label = diagnosis_to_malignant(row['diagnosis'])
        out = {
            'clinic': clinic_img, 'derm': derm_img,
            'diagnosis': torch.tensor(diag_label, dtype=torch.long),
            'malignant': torch.tensor(malignant_label, dtype=torch.long),
        }
        for col in CRITERIA_COLUMNS:
            out[col] = torch.tensor(criterion_to_index(col, row[col]), dtype=torch.long)
        return out

In [ ]:
# ---- model.py ----
class MultiTaskDerm7pt(nn.Module):
    def __init__(self, backbone_name, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        with torch.no_grad():
            feat_dim = self.backbone(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).shape[1]
        fused_dim = feat_dim * 2
        self.dropout = nn.Dropout(dropout)
        self.diagnosis_head = nn.Linear(fused_dim, len(DIAGNOSIS_CLASSES))
        self.malignant_head = nn.Linear(fused_dim, 2)
        self.criteria_heads = nn.ModuleDict({n: nn.Linear(fused_dim, c) for n, c in CRITERIA_NUM_CLASSES.items()})
        task_names = ['diagnosis', 'malignant'] + list(CRITERIA_NUM_CLASSES.keys())
        self.log_vars = nn.ParameterDict({n: nn.Parameter(torch.zeros(())) for n in task_names})

    def forward(self, clinic, derm):
        fc = self.backbone(clinic)
        fd = self.backbone(derm)
        fused = self.dropout(torch.cat([fc, fd], dim=1))
        out = {'diagnosis': self.diagnosis_head(fused), 'malignant': self.malignant_head(fused)}
        for name, head in self.criteria_heads.items():
            out[name] = head(fused)
        return out

    def multitask_loss(self, outputs, targets):
        total = 0.0
        for name, log_var in self.log_vars.items():
            task_loss = F.cross_entropy(outputs[name], targets[name])
            total = total + torch.exp(-log_var) * task_loss + log_var
        return total

In [ ]:
# ---- engine.py ----
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TASK_NAMES = ['diagnosis', 'malignant'] + list(CRITERIA_NUM_CLASSES.keys())

def build_transforms(img_size, train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)), transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15), transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)), transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def expected_calibration_error(probs, labels, n_bins=10):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct = (predictions == labels).astype(np.float64)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(labels)) * abs(correct[mask].mean() - confidences[mask].mean())
    return float(ece)

def train_one_epoch(model, raw_model, loader, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        clinic = batch['clinic'].to(DEVICE)
        derm = batch['derm'].to(DEVICE)
        targets = {k: batch[k].to(DEVICE) for k in TASK_NAMES}
        optimizer.zero_grad()
        outputs = model(clinic, derm)
        loss = raw_model.multitask_loss(outputs, targets)
        loss.backward()
        optimizer.step()
        bs = clinic.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / max(n, 1)

@torch.no_grad()
def evaluate(model, raw_model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    all_logits = {k: [] for k in TASK_NAMES}
    all_labels = {k: [] for k in TASK_NAMES}
    for batch in loader:
        clinic = batch['clinic'].to(DEVICE)
        derm = batch['derm'].to(DEVICE)
        targets = {k: batch[k].to(DEVICE) for k in TASK_NAMES}
        outputs = model(clinic, derm)
        loss = raw_model.multitask_loss(outputs, targets)
        bs = clinic.size(0)
        total_loss += loss.item() * bs
        n += bs
        for k in TASK_NAMES:
            all_logits[k].append(outputs[k].cpu())
            all_labels[k].append(targets[k].cpu())
    metrics = {'val_loss': total_loss / max(n, 1)}
    for k in TASK_NAMES:
        logits = torch.cat(all_logits[k]).numpy()
        labels = torch.cat(all_labels[k]).numpy()
        probs = F.softmax(torch.from_numpy(logits), dim=1).numpy()
        preds = probs.argmax(axis=1)
        metrics[f'{k}_acc'] = float((preds == labels).mean())
        metrics[f'{k}_macro_f1'] = float(f1_score(labels, preds, average='macro', zero_division=0))
        if k == 'diagnosis':
            metrics['diagnosis_ece'] = expected_calibration_error(probs, labels)
    return metrics

In [ ]:
# ---- backbone tier: THIS NOTEBOOK ONLY RUNS 'lightweight' ----
BACKBONE_TIERS = {
    'lightweight': ['mobilenetv3_small_100', 'mobilenetv3_large_100', 'efficientnet_b0', 'resnet18'],
}

In [ ]:
# ---- the sweep: train every backbone, every tier, log a leaderboard ----
train_tf = build_transforms(IMG_SIZE, train=True)
valid_tf = build_transforms(IMG_SIZE, train=False)
train_ds = Derm7ptDataset(INPUT_DIR, 'train', transform=train_tf)
valid_ds = Derm7ptDataset(INPUT_DIR, 'valid', transform=valid_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f'train={len(train_ds)}  valid={len(valid_ds)}')

results = []
overall_best = {'diagnosis_acc': -1.0, 'backbone': None}

for tier_name, backbones in BACKBONE_TIERS.items():
    print(f'\n========== TIER: {tier_name} ==========')
    tier_best_acc = -1.0
    for backbone_name in backbones:
        print(f'\n--- {backbone_name} ---')
        t0 = time.time()
        model = MultiTaskDerm7pt(backbone_name, pretrained=True).to(DEVICE)
        raw_model = model
        if MULTI_GPU:
            model = nn.DataParallel(model)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

        best_acc, best_metrics = -1.0, None
        for epoch in range(1, EPOCHS_PER_BACKBONE + 1):
            train_loss = train_one_epoch(model, raw_model, train_loader, optimizer)
            metrics = evaluate(model, raw_model, valid_loader)
            if metrics['diagnosis_acc'] > best_acc:
                best_acc = metrics['diagnosis_acc']
                best_metrics = metrics
                torch.save(raw_model.state_dict(), f'{OUTPUT_DIR}/checkpoints/{backbone_name}_best.pt')
            if epoch % 5 == 0 or epoch == EPOCHS_PER_BACKBONE:
                print(f'  epoch {epoch}/{EPOCHS_PER_BACKBONE}  train_loss={train_loss:.4f}  '
                      f'val_diag_acc={metrics["diagnosis_acc"]:.4f}  ece={metrics["diagnosis_ece"]:.4f}')

        dt = time.time() - t0
        n_params = sum(p.numel() for p in raw_model.parameters())
        row = {'tier': tier_name, 'backbone': backbone_name, 'params': n_params,
               'train_time_s': round(dt, 1), **best_metrics}
        results.append(row)
        tier_best_acc = max(tier_best_acc, best_acc)
        if best_acc > overall_best['diagnosis_acc']:
            overall_best = {'diagnosis_acc': best_acc, 'backbone': backbone_name, 'tier': tier_name}
        print(f'  -> best diagnosis_acc={best_acc:.4f}  ({dt:.1f}s, {n_params:,} params)')

    print(f'Tier {tier_name} best: {tier_best_acc:.4f}')

train=413  valid=203

========== TIER: lightweight ==========

--- mobilenetv3_small_100 ---


  epoch 5/15  train_loss=4.7945  val_diag_acc=0.4581  ece=0.1626
  epoch 10/15  train_loss=2.4215  val_diag_acc=0.5468  ece=0.1900
  epoch 15/15  train_loss=0.9659  val_diag_acc=0.5517  ece=0.2736
  -> best diagnosis_acc=0.5517  (240.9s, 1,610,070 params)

--- mobilenetv3_large_100 ---

  epoch 5/15  train_loss=4.6841  val_diag_acc=0.6355  ece=0.1411
  epoch 10/15  train_loss=1.4559  val_diag_acc=0.6305  ece=0.1276
  epoch 15/15  train_loss=0.3282  val_diag_acc=0.6158  ece=0.1915
  -> best diagnosis_acc=0.6502  (112.2s, 4,317,286 params)

--- efficientnet_b0 ---

  epoch 5/15  train_loss=4.5254  val_diag_acc=0.5468  ece=0.0811
  epoch 10/15  train_loss=1.8483  val_diag_acc=0.6158  ece=0.0621
  epoch 15/15  train_loss=0.4241  val_diag_acc=0.6158  ece=0.1075
  -> best diagnosis_acc=0.6305  (118.0s, 4,122,802 params)

--- resnet18 ---

  epoch 5/15  train_loss=7.6750  val_diag_acc=0.3448  ece=0.

In [ ]:
# ---- leaderboard ----
df = pd.DataFrame(results).sort_values('diagnosis_acc', ascending=False)
df.to_csv(f'{OUTPUT_DIR}/sweep_results_lightweight.csv', index=False)
print(f"\nBEST OVERALL: {overall_best['backbone']} ({overall_best['tier']} tier), "
      f"diagnosis_acc={overall_best['diagnosis_acc']:.4f}")
print(f"Checkpoint saved at: {OUTPUT_DIR}/checkpoints/{overall_best['backbone']}_best.pt")
df[['tier', 'backbone', 'params', 'diagnosis_acc', 'diagnosis_macro_f1', 'diagnosis_ece',
    'malignant_acc', 'train_time_s']]

BEST OVERALL: mobilenetv3_large_100 (lightweight tier), diagnosis_acc=0.6502
Checkpoint saved at: /kaggle/working/checkpoints/mobilenetv3_large_100_best.pt

	tier	backbone	params	diagnosis_acc	diagnosis_macro_f1	diagnosis_ece	malignant_acc	train_time_s
1	lightweight	mobilenetv3_large_100	4317286	0.650246	0.283732	0.093087	0.822660	112.2
2	lightweight	efficientnet_b0	4122802	0.630542	0.284638	0.109625	0.793103	118.0
3	lightweight	resnet18	11222646	0.566502	0.228599	0.073012	0.773399	107.4
0	lightweight	mobilenetv3_small_100	1610070	0.551724	0.284888	0.175757	0.748768	240.9